# 06 — Forecast KPIs

**Reference:** Vandeput, N. (2021). *Data Science for Supply Chain Forecasting* (2nd ed.), Chapter 2.

A forecast without a measurement framework is useless. This notebook implements
all major forecast KPIs and builds a **model comparison utility** used throughout
the rest of this repo.

| KPI | Full Name | Sensitive to | Best For |
|-----|-----------|-------------|----------|
| MAE | Mean Absolute Error | Magnitude | General use |
| RMSE | Root Mean Squared Error | Large errors | Penalising spikes |
| MAPE | Mean Absolute % Error | Scale | Cross-SKU comparison |
| Bias | Mean Error | Direction | Detecting systematic over/under |
| SMAPE | Symmetric MAPE | Asymmetry | Low-volume items |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. KPI Functions — Reusable Module

In [ ]:
def forecast_kpis(actual, forecast):
    """
    Compute all standard forecast KPIs.

    Parameters
    ----------
    actual   : array-like — actual demand values
    forecast : array-like — forecast values (same length)

    Returns
    -------
    dict of KPI name -> value
    """
    actual   = np.array(actual, dtype=float)
    forecast = np.array(forecast, dtype=float)

    # Mask NaN pairs
    valid = ~(np.isnan(actual) | np.isnan(forecast))
    a, f = actual[valid], forecast[valid]
    errors = a - f

    mae   = np.mean(np.abs(errors))
    rmse  = np.sqrt(np.mean(errors ** 2))
    bias  = np.mean(errors)

    # MAPE — exclude zero actual values (division by zero)
    nonzero = a != 0
    mape  = np.mean(np.abs(errors[nonzero] / a[nonzero])) * 100

    # SMAPE
    smape = np.mean(2 * np.abs(errors) / (np.abs(a) + np.abs(f) + 1e-8)) * 100

    return {
        'MAE':   round(mae,  2),
        'RMSE':  round(rmse, 2),
        'Bias':  round(bias, 2),
        'MAPE':  round(mape, 2),
        'SMAPE': round(smape, 2),
        'n':     int(valid.sum())
    }


def compare_models(results_dict):
    """
    Compare multiple models by their KPIs.

    Parameters
    ----------
    results_dict : dict of {model_name: (actual, forecast)}

    Returns
    -------
    pd.DataFrame — one row per model
    """
    rows = []
    for name, (actual, forecast) in results_dict.items():
        kpis = forecast_kpis(actual, forecast)
        kpis['Model'] = name
        rows.append(kpis)
    df = pd.DataFrame(rows).set_index('Model')
    return df[['MAE', 'RMSE', 'Bias', 'MAPE', 'SMAPE', 'n']]

## 2. Compare All Part I Models on the Same Dataset

In [ ]:
# Import our models from previous notebooks
# (in practice, refactor these into a utils.py module)

def moving_average(d, extra_periods=0, n=3):
    cols = len(d)
    d = np.append(d, [np.nan] * extra_periods)
    f = np.full(cols + extra_periods, np.nan)
    for t in range(n, cols):
        f[t] = np.mean(d[t - n:t])
    f[cols:] = np.mean(d[cols-n:cols])
    return np.array(d), np.array(f)

def exp_smoothing(d, extra_periods=0, alpha=0.4):
    cols = len(d)
    d = np.append(d, [np.nan] * extra_periods)
    f = np.full(cols + extra_periods, np.nan)
    f[1] = d[0]
    for t in range(2, cols):
        f[t] = alpha * d[t-1] + (1 - alpha) * f[t-1]
    f[cols:] = f[cols-1]
    return np.array(d), np.array(f)

# Simulate shared dataset
np.random.seed(42)
demand = np.random.normal(1000, 100, 48)

a_ma, f_ma = moving_average(demand, n=3)
a_es, f_es = exp_smoothing(demand, alpha=0.4)

comparison = compare_models({
    'Moving Average (n=3)': (a_ma, f_ma),
    'Exp Smoothing (α=0.4)': (a_es, f_es),
})

print('=== Model Comparison ===')
print(comparison.to_string())

## 3. Bias Deep Dive — The Most Overlooked KPI

In [ ]:
# Bias = systematic over or under forecasting
# Even a low MAE can hide a dangerous bias

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

errors_ma = a_ma - f_ma
errors_es = a_es - f_es

for ax, errors, label, color in zip(
    axes,
    [errors_ma, errors_es],
    ['Moving Average', 'Exp Smoothing'],
    ['#185FA5', '#D85A30']
):
    valid_errors = errors[~np.isnan(errors)]
    ax.hist(valid_errors, bins=15, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.axvline(np.mean(valid_errors), color='red', linestyle='-',
               linewidth=1.5, label=f'Bias: {np.mean(valid_errors):.1f}')
    ax.set_title(f'{label} — Error Distribution')
    ax.set_xlabel('Forecast Error (actual - forecast)')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Bias Analysis — Are We Systematically Over or Under Forecasting?',
             fontsize=12)
plt.tight_layout()
plt.show()

## Key Takeaways

- **MAE** is your primary accuracy metric — robust and interpretable
- **Bias** is your primary health metric — tells you if your model has a systematic problem
- **MAPE** is useful for comparing across products with different scales
- Always check all three before declaring a model good

This `forecast_kpis()` and `compare_models()` utility will be used in every
subsequent notebook as the standard evaluation framework.

**Next:** Part II — Machine Learning (`02_machine_learning/01_decision_trees.ipynb`)
